# Figure 3

Mutual 10-NN alignment and linear CKA between the twelve image models without language supervision and six encoders: CLIP-B and SigLIP2-B given the word images, three Qwen3 text encoders given the words as text, and w2v-BERT 2.0 given them as speech.

Inputs: `scripts/analysis/figure3_language_alignment.py` (`results/language_alignment.parquet`) and `scripts/analysis/table1_models.py` (`results/retrieval.parquet`, for the one-edit word column of Table 1).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.font_manager import FontProperties
from matplotlib.textpath import TextPath

from src.features.speech import SPEECH_MODELS
from src.features.text import TEXT_MODELS
from src.models import DEFAULT_SWEEP, FAMILY_OF, MODELS, PRETTY
from src.tau import word_order
from src.utils import setup_notebook, viz
from src.utils.results import read

setup_notebook()

ALIGNMENT = read("language_alignment")
RETRIEVAL = read("retrieval")

#: The two measures, upper-left triangle first.
MEASURES = ("mutual_knn", "cka")
MEASURE_TITLE = {"mutual_knn": "Mutual 10-NN alignment", "cka": "Linear CKA"}

IMAGE_ROWS = ["clip-b", "siglip2-b"]
TEXT_ROWS = list(TEXT_MODELS)
SPEECH_ROWS = list(SPEECH_MODELS)
ROWS = IMAGE_ROWS + TEXT_ROWS + SPEECH_ROWS
COLUMNS = [m for m in DEFAULT_SWEEP if m not in IMAGE_ROWS]
#: The input each row encoder receives.
GROUP_OF = (dict.fromkeys(IMAGE_ROWS, "Image") | dict.fromkeys(TEXT_ROWS, "Text")
            | dict.fromkeys(SPEECH_ROWS, "Speech"))

VALUE = ALIGNMENT.pivot_table(index=["measure", "reference"], columns="model",
                              values="value")
with pd.option_context("display.float_format", "{:.3f}".format):
    for name in MEASURES:
        print(MEASURE_TITLE[name])
        display(VALUE.loc[name].loc[ROWS, COLUMNS].rename(index=PRETTY, columns=PRETTY))

## Chance and the length null

The null relabels a row encoder's words among words with the same number of letters, with the same relabelings for every cell.

In [ ]:
N_WORDS = len(word_order())
N_PERM = int(ALIGNMENT.n_perm.iloc[0])
#: The smallest p the relabelings allow, reached when every relabeling scores
#: below the observed value.
FLOOR = 1 / (N_PERM + 1)

mutual_chance = ALIGNMENT[ALIGNMENT.measure.eq("mutual_knn")].chance.iloc[0]
cka_chance = ALIGNMENT[ALIGNMENT.measure.eq("cka")].chance
print(f"words: {N_WORDS:,}")
print(f"Mutual 10-NN alignment chance: 10/{N_WORDS - 1:,} = {mutual_chance:.3f}")
print(f"Linear CKA chance: {cka_chance.min():.3f} to {cka_chance.max():.3f}")
print(f"relabelings: {N_PERM:,}")
exceeds = np.isclose(ALIGNMENT.p, FLOOR)
untrained = ALIGNMENT.model.eq("vit-random-b")
print(f"cells above every relabeling, p = 1/{N_PERM + 1:,}: {exceeds.sum()} of "
      f"{len(ALIGNMENT)}, and {(exceeds & untrained).sum()} of the {untrained.sum()} "
      f"cells of the untrained ViT-B")
print(f"cells at Benjamini-Hochberg adjusted p = 1/{N_PERM + 1:,}, over the "
      f"{len(ROWS) * len(COLUMNS)} cells of each measure: "
      f"{np.isclose(ALIGNMENT.p_bh, FLOOR).sum()} of {len(ALIGNMENT)}")

## Values cited in Section 6

In [ ]:
LARGE = ["dinov3-h", "dinov3-7b"]
#: For each measure, the column with the largest value in every row.
ROW_LARGEST = {name: VALUE.loc[name].loc[ROWS, COLUMNS].idxmax(axis=1)
               for name in MEASURES}


def cells(name: str, rows, columns) -> np.ndarray:
    return VALUE.loc[name].loc[rows, columns].to_numpy().ravel()


text_large = cells("mutual_knn", TEXT_ROWS, LARGE)
print(f"Mutual 10-NN alignment, text rows, DINOv3-H and DINOv3-7B: "
      f"{text_large.min():.2f} to {text_large.max():.2f} "
      f"({text_large.min():.0%} to {text_large.max():.0%})")
others = [m for m in COLUMNS if m not in ("dinov3-l", *LARGE)]
text_others = cells("mutual_knn", TEXT_ROWS, others).max()
print(f"Mutual 10-NN alignment, text rows, the {len(others)} models other than "
      f"DINOv3-L, -H and -7B ({', '.join(PRETTY[m] for m in others)}): at most "
      f"{text_others:.2f} ({text_others:.0%})")

#: One-edit word top-10 accuracy as in Table 1, each anchor queried once.
one_edit = (RETRIEVAL[RETRIEVAL.relation.eq("spelling_twin")
                      & RETRIEVAL.direction.eq("forward")].set_index("model").rate10)
self_supervised = [m for m in COLUMNS if MODELS[m].kind in ("dino", "mae")]
most = one_edit[self_supervised].idxmax()
print(f"one-edit word top-10 accuracy in Table 1, largest of the {len(self_supervised)} "
      f"self-supervised models: {PRETTY[most]} {one_edit[most]:.3f}")

print("largest value in each row:")
for name in MEASURES:
    for row in ROWS:
        column = ROW_LARGEST[name][row]
        print(f"  {MEASURE_TITLE[name]}, {PRETTY[row]}: {PRETTY[column]} "
              f"{VALUE.loc[name].loc[row, column]:.2f}")

print("two largest values in each text row:")
for name in MEASURES:
    for row in TEXT_ROWS:
        top = VALUE.loc[name].loc[row, COLUMNS].nlargest(2)
        print(f"  {MEASURE_TITLE[name]}, {PRETTY[row]}: "
              + ", ".join(f"{PRETTY[m]} {v:.2f}" for m, v in top.items()))

for name in MEASURES:
    table = VALUE.loc[name].loc[ROWS, COLUMNS]
    above = (table.loc[IMAGE_ROWS].min() > table.loc[TEXT_ROWS].max()).sum()
    print(f"{MEASURE_TITLE[name]}: columns where CLIP-B and SigLIP2-B both exceed "
          f"every text row: {above} of {len(COLUMNS)}")

for model in ("mae-h", "dinov3-7b"):
    print(f"Linear CKA, SigLIP2-B, {PRETTY[model]}: "
          f"{VALUE.loc['cka'].loc['siglip2-b', model]:.2f}")

mutual = VALUE.loc["mutual_knn"]
below = (mutual.loc[TEXT_ROWS, "dinov3-l"] < mutual.loc[TEXT_ROWS, LARGE].min(axis=1)).sum()
print(f"Mutual 10-NN alignment, text rows where DINOv3-L is below DINOv3-H and "
      f"DINOv3-7B: {below} of {len(TEXT_ROWS)}")

## The figure

The upper-left triangle of a cell shows mutual 10-NN alignment and the lower-right triangle shows linear CKA. Each measure has one logarithmic color scale, and an outline marks the largest value of each measure in a row.

In [ ]:
WIDTH_PT = viz.TEXT_WIDTH_IN * viz.PT_PER_INCH
#: A cell's height as a fraction of its width.
CELL_ASPECT = 0.70
#: The drawing is laid out from the top on a page this tall, then trimmed to
#: the bottom of the key plus ``BOTTOM_PT``.
DRAFT_HEIGHT_PT = 400.0
BOTTOM_PT = 1.5

#: Row names on the left and row groups on the right of the grid.
GUTTER_PT = 50.0
GROUP_LABEL_PT = 15.0
#: White space between neighboring cells, between model families, and between
#: row groups.
CELL_GAP_PT = 1.2
FAMILY_GAP_PT = 6.0
GROUP_GAP_PT = 5.0

TOP_PT = 1.0
HEADER_PT = 25.0
KEY_GAP_PT = 5.0
#: Baseline to baseline, a key title to its subtitle.
KEY_LINE_GAP_PT = 8.5
KEY_BAR_LENGTH_PT = 104.0
KEY_BAR_HEIGHT_PT = 3.4
KEY_BAR_GAP_PT = 2.4
#: The outline around the largest triangle of each measure in a row, drawn
#: inside the triangle.
OUTLINE_WIDTH_PT = 1.0

KEY_SUBTITLE = {"mutual_knn": "10 nearest neighbors",
                "cka": "whole similarity matrix"}

#: A measure's color scale runs from its smallest cell, rounded down to a
#: multiple of the first step and never below it, to its largest cell rounded
#: up to a multiple of the second.
SCALE_FLOOR_STEP = 0.005
SCALE_TOP_STEP = 0.1
SCALE_LIMITS = {
    name: (max(round(float(np.floor(part.value.min() / SCALE_FLOOR_STEP))
                     * SCALE_FLOOR_STEP, 3), SCALE_FLOOR_STEP),
           round(float(np.ceil(part.value.max() / SCALE_TOP_STEP))
                 * SCALE_TOP_STEP, 3))
    for name, part in ALIGNMENT.groupby("measure")}
#: Where each key bar starts.
KEY_FROM = {"mutual_knn": 0.0, "cka": 0.0}
#: Key ticks every 0.1 on an even axis of values.
SCALE_TICKS = {name: [round(t, 2) for t in
                      np.arange(np.ceil(KEY_FROM[name] / 0.1 - 1e-9) * 0.1,
                                high + 1e-9, 0.1)]
               for name, (_, high) in SCALE_LIMITS.items()}

In [ ]:
def proportion(value: float, places: int) -> str:
    """A proportion without its leading zero, as ``.13``."""
    return f"{value:.{places}f}".removeprefix("0")


def cell_size() -> tuple[float, float]:
    """The width and the height of a cell."""
    families = [FAMILY_OF[m] for m in COLUMNS]
    breaks = sum(a != b for a, b in zip(families, families[1:]))
    wide = ((WIDTH_PT - GUTTER_PT - GROUP_LABEL_PT - breaks * FAMILY_GAP_PT)
            / len(COLUMNS))
    return wide, wide * CELL_ASPECT


def column_plan(wide: float) -> list[tuple[str, str, float]]:
    """``(model, family, left edge)`` for every column."""
    out, x, previous = [], GUTTER_PT, None
    for model in COLUMNS:
        family = FAMILY_OF[model]
        if previous is not None and family != previous:
            x += FAMILY_GAP_PT
        out.append((model, family, x))
        x += wide
        previous = family
    return out


def row_plan(high: float) -> list[tuple[str, str, float]]:
    """``(row encoder, group, top edge)`` for every row."""
    out, y, previous = [], TOP_PT + HEADER_PT, None
    for reference in ROWS:
        group = GROUP_OF[reference]
        if previous is not None and group != previous:
            y += GROUP_GAP_PT
        out.append((reference, group, y))
        y += high
        previous = group
    return out


def color_scale() -> dict:
    """``{measure: (ramp, logarithmic norm)}``."""
    return {"mutual_knn": viz.neighbor_alignment_scale(*SCALE_LIMITS["mutual_knn"]),
            "cka": viz.cka_scale(*SCALE_LIMITS["cka"])}


def triangles(x: float, y: float, wide: float, high: float):
    """The upper-left and lower-right triangles of a cell, y pointing down."""
    upper = ((x, y), (x + wide, y), (x, y + high))
    lower = ((x + wide, y), (x + wide, y + high), (x, y + high))
    return upper, lower


def label(ax, x, y, text, *, size, colour, ha="center", va="baseline", rotation=0):
    return ax.text(x, y, text, ha=ha, va=va, fontsize=size, color=colour,
                   rotation=rotation, rotation_mode="anchor")


def inset(triangle, by: float) -> np.ndarray:
    """A triangle with every edge moved inward by ``by``.

    Scaling a triangle about its incenter moves all three edges inward by the
    same distance. The incenter is the vertices weighted by the opposite sides,
    and the inradius is twice the area over the perimeter.
    """
    points = np.asarray(triangle, float)
    sides = np.linalg.norm(np.roll(points, -1, 0) - np.roll(points, 1, 0),
                           axis=1)
    centre = (sides[:, None] * points).sum(0) / sides.sum()
    (ux, uy), (vx, vy) = points[1] - points[0], points[2] - points[0]
    area = abs(ux * vy - uy * vx) / 2
    radius = 2 * area / sides.sum()
    return centre + (points - centre) * (radius - by) / radius


def text_width(fig, ax, texts) -> float:
    """The widest of some texts, in points."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_points = ax.transData.inverted()
    widths = []
    for t in texts:
        extent = t.get_window_extent(renderer)
        (x0, _), (x1, _) = to_points.transform([[extent.x0, extent.y0],
                                                [extent.x1, extent.y1]])
        widths.append(x1 - x0)
    return max(widths)


def digit_extent(size: float) -> tuple[float, float]:
    """How far the numerals rise above their baseline and fall below it, in points."""
    box = TextPath((0, 0), "0123456789.", size=size, prop=FontProperties()).get_extents()
    return float(box.y1), float(max(-box.y0, 0.0))


def cap_height(size: float) -> float:
    """The height of a capital letter at ``size`` points, in points."""
    return float(TextPath((0, 0), "H", size=size, prop=FontProperties()).get_extents().height)

In [ ]:
def draw_grid(ax, scale):
    wide, high = cell_size()
    rise, fall = digit_extent(viz.TEXT_SIZE)
    inner_wide, inner_high = wide - CELL_GAP_PT, high - CELL_GAP_PT
    polygons, fills, diagonals, outlines = [], [], [], []
    for reference, _, top in row_plan(high):
        for model, _, left in column_plan(wide):
            x, y = left + CELL_GAP_PT / 2, top + CELL_GAP_PT / 2
            for name, shape, spot in zip(MEASURES,
                                         triangles(x, y, inner_wide, inner_high),
                                         ((0.28, 0.32), (0.72, 0.70))):
                value = float(VALUE.loc[name].loc[reference, model])
                ramp, norm = scale[name]
                polygons.append(shape)
                fills.append(ramp(norm(value)))
                if ROW_LARGEST[name][reference] == model:
                    outlines.append(inset(shape, OUTLINE_WIDTH_PT / 2 + 0.1))
                # The numeral color follows the printed value, and the numerals
                # are centered on their own ink rather than on the line box.
                shown = round(value, 2)
                background = ramp(norm(max(shown, SCALE_LIMITS[name][0])))
                base = y + spot[1] * inner_high + (rise - fall) / 2
                label(ax, x + spot[0] * inner_wide, base, proportion(shown, 2),
                      size=viz.TEXT_SIZE,
                      colour=(viz.WHITE if viz.luminance(background) < 0.229
                              else viz.INK))
            diagonals.append(((x, y + inner_high), (x + inner_wide, y)))
    ax.add_collection(PolyCollection(polygons, facecolors=fills,
                                     edgecolors="none", antialiased=True))
    ax.add_collection(LineCollection(diagonals, colors=viz.WHITE,
                                     linewidths=0.8))
    ax.add_collection(PolyCollection(outlines, facecolors="none",
                                     edgecolors=viz.INK,
                                     linewidths=OUTLINE_WIDTH_PT,
                                     joinstyle="round"))


def draw_headers(ax):
    wide, high = cell_size()
    columns = column_plan(wide)
    for family in dict.fromkeys(f for _, f, _ in columns):
        span = [left for _, f, left in columns if f == family]
        left, right = span[0], span[-1] + wide
        label(ax, (left + right) / 2, TOP_PT + 8.0, family,
              size=viz.TITLE_SIZE, colour=viz.INK)
        # A single column keeps more of its width, so its rule is not a stub.
        margin = 3.0 if len(span) > 1 else 1.0
        ax.plot([left + margin, right - margin], [TOP_PT + 11.5] * 2,
                color=viz.CONTROL, lw=0.45, solid_capstyle="butt")
    for model, family, left in columns:
        suffix = PRETTY[model].removeprefix(family).strip(" -")
        label(ax, left + wide / 2, TOP_PT + 20.5, suffix,
              size=viz.TEXT_SIZE, colour=viz.CONTROL)
    rows = row_plan(high)
    grid_right = columns[-1][2] + wide
    for reference, _, top in rows:
        label(ax, GUTTER_PT - 4.0, top + high / 2, PRETTY[reference],
              size=viz.TEXT_SIZE, colour=viz.CONTROL, ha="right", va="center")
    for group in dict.fromkeys(g for _, g, _ in rows):
        span = [top for _, g, top in rows if g == group]
        top, bottom = span[0], span[-1] + high
        ax.plot([grid_right + 4.0] * 2, [top + 1.5, bottom - 1.5],
                color=viz.CHANCE, lw=0.45, solid_capstyle="butt")
        label(ax, grid_right + 9.5, (top + bottom) / 2, group,
              size=viz.TEXT_SIZE, colour=viz.CONTROL, va="center", rotation=-90)

In [ ]:
def draw_key_item(fig, ax, name, scale, x, block_top):
    """A small cell with one triangle filled, then the measure's name and subtitle.

    The cell is as tall as the two lines of text, from the top of the title's
    capitals to the subtitle's baseline. Returns the artists drawn and the
    item's right edge.
    """
    high = cap_height(viz.TITLE_SIZE) + KEY_LINE_GAP_PT
    wide = high / CELL_ASPECT
    upper, lower = triangles(x, block_top, wide, high)
    filled = 0 if name == "mutual_knn" else 1
    glyph = ax.add_collection(PolyCollection(
        [upper, lower],
        facecolors=[scale[name][0](0.62) if i == filled else viz.FAINT
                    for i in range(2)],
        edgecolors="none"))
    (diagonal,) = ax.plot([x, x + wide], [block_top + high, block_top],
                          color=viz.WHITE, lw=0.8)
    text_x = x + wide + 4.5
    title = block_top + cap_height(viz.TITLE_SIZE)
    texts = [label(ax, text_x, title, MEASURE_TITLE[name],
                   size=viz.TITLE_SIZE, colour=viz.INK, ha="left"),
             label(ax, text_x, title + KEY_LINE_GAP_PT, KEY_SUBTITLE[name],
                   size=viz.TEXT_SIZE, colour=viz.CONTROL, ha="left")]
    return [glyph, diagonal, *texts], text_x + text_width(fig, ax, texts)


def draw_key_scale(ax, scale, left, bar_top) -> float:
    """Both scales as stacked bars on even axes of values, ticked outward.

    Each bar runs from ``KEY_FROM`` to the top of its scale at an even rate of
    value. The upper bar is the upper-left triangle's measure and is ticked
    above; the lower bar is ticked below. Returns the bottom.
    """
    steps = 200
    width = KEY_BAR_LENGTH_PT / steps
    for row, name in enumerate(MEASURES):
        ramp, norm = scale[name]
        low, high = SCALE_LIMITS[name]
        y = bar_top + row * (KEY_BAR_HEIGHT_PT + KEY_BAR_GAP_PT)
        start = KEY_FROM[name]
        values = start + (np.arange(steps) + 0.5) / steps * (high - start)
        ax.add_collection(PolyCollection(
            [((left + i * width, y), (left + (i + 1) * width, y),
              (left + (i + 1) * width, y + KEY_BAR_HEIGHT_PT),
              (left + i * width, y + KEY_BAR_HEIGHT_PT))
             for i in range(steps)],
            facecolors=[ramp(norm(max(v, low))) for v in values],
            edgecolors="face", linewidths=0.3))
        for tick in SCALE_TICKS[name]:
            x = left + KEY_BAR_LENGTH_PT * (tick - start) / (high - start)
            text = "0" if tick == 0 else proportion(tick, 1)
            if row == 0:
                ax.plot([x, x], [y - 1.6, y], color=viz.CONTROL, lw=0.45)
                label(ax, x, y - 2.6, text, size=viz.TEXT_SIZE,
                      colour=viz.CONTROL)
            else:
                edge = y + KEY_BAR_HEIGHT_PT
                ax.plot([x, x], [edge, edge + 1.6], color=viz.CONTROL, lw=0.45)
                label(ax, x, edge + 2.6 + cap_height(viz.TEXT_SIZE), text,
                      size=viz.TEXT_SIZE, colour=viz.CONTROL)
    return (bar_top + 2 * KEY_BAR_HEIGHT_PT + KEY_BAR_GAP_PT + 2.6
            + cap_height(viz.TEXT_SIZE) + 2.0)


def draw_key(fig, ax, scale) -> float:
    """One line under the grid, as wide as the grid, with a measure at each end.

    The text blocks and the pair of bars share one vertical center. Returns
    the bottom of the key.
    """
    wide, high = cell_size()
    columns = column_plan(wide)
    grid_left, grid_right = columns[0][2], columns[-1][2] + wide
    top = row_plan(high)[-1][2] + high + KEY_GAP_PT
    bars = 2 * KEY_BAR_HEIGHT_PT + KEY_BAR_GAP_PT
    bar_top = top + cap_height(viz.TEXT_SIZE) + 2.6
    block = cap_height(viz.TITLE_SIZE) + KEY_LINE_GAP_PT
    block_top = bar_top + bars / 2 - block / 2
    _, left_end = draw_key_item(fig, ax, "mutual_knn", scale, grid_left,
                                block_top)
    # The right item is drawn once to measure it, then drawn again flush right.
    trial, width = draw_key_item(fig, ax, "cka", scale, 0.0, block_top)
    for artist in trial:
        artist.remove()
    right_start = grid_right - width
    draw_key_item(fig, ax, "cka", scale, right_start, block_top)
    centre = (left_end + right_start) / 2
    bottom = draw_key_scale(ax, scale, centre - KEY_BAR_LENGTH_PT / 2, bar_top)
    return max(bottom, block_top + block + 2.0)


def draw_figure():
    scale = color_scale()
    fig = plt.figure(figsize=(WIDTH_PT / viz.PT_PER_INCH,
                              DRAFT_HEIGHT_PT / viz.PT_PER_INCH))
    ax = fig.add_axes((0, 0, 1, 1))
    ax.set_xlim(0, WIDTH_PT)
    ax.set_ylim(DRAFT_HEIGHT_PT, 0)
    ax.set_axis_off()
    draw_grid(ax, scale)
    draw_headers(ax)
    bottom = draw_key(fig, ax, scale)
    # One data unit is one point, so the page is trimmed by shortening the
    # figure and the axes together.
    height = round(bottom + BOTTOM_PT, 3)
    fig.set_size_inches(WIDTH_PT / viz.PT_PER_INCH, height / viz.PT_PER_INCH)
    ax.set_ylim(height, 0)
    return fig

In [ ]:
fig = draw_figure()
viz.save(fig, "figure3_language_alignment")
plt.show()